In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/varshagujrathi014@gmail.com/FMCG-pipeline/setup/utilities

In [0]:
dbutils.widgets.text("catalog","fmcg","catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://pipeline-sportsbar/customers/customers.csv"
print(base_path)

s3://pipeline-sportsbar/customers/customers.csv


In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", f.current_timestamp())
    .select("*", f.col("_metadata.file_name").alias("_metadata_file_name"), f.col("_metadata.file_size").alias("_metadata_file_size"))

)
display(df.limit(10))

customer_id,customer_name,city,read_timestamp,_metadata_file_name,_metadata_file_size
789201,FitFuel Market,Bengaluru,2026-09-19T17:17:20.830Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-09-19T17:17:20.830Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-09-19T17:17:20.830Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-09-19T17:17:20.830Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-09-19T17:17:20.830Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-09-19T17:17:20.830Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-09-19T17:17:20.830Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-09-19T17:17:20.830Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-09-19T17:17:20.830Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-09-19T17:17:20.830Z,customers.csv,1404


In [0]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- _metadata_file_name: string (nullable = false)
 |-- _metadata_file_size: long (nullable = false)



In [0]:
df.write\
    .format("delta")\
    .option("overwrite.enableChangeDataFeed","true") \
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")


### Silver Processing ###

In [0]:
df.bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df.bronze.show(10)

+-----------+--------------------+---------+--------------------+-------------------+-------------------+
|customer_id|       customer_name|     city|      read_timestamp|_metadata_file_name|_metadata_file_size|
+-----------+--------------------+---------+--------------------+-------------------+-------------------+
|     789201|      FitFuel Market|Bengaluru|2026-09-19 17:23:...|      customers.csv|               1404|
|     789202|      FitFuel Market|Hyderabad|2026-09-19 17:23:...|      customers.csv|               1404|
|     789203|      FitFuel Market|New Delhi|2026-09-19 17:23:...|      customers.csv|               1404|
|     789301|Athlete's Choice ...|Bengaluru|2026-09-19 17:23:...|      customers.csv|               1404|
|     789303|Athlete's Choice ...|New Delhi|2026-09-19 17:23:...|      customers.csv|               1404|
|     789101|     Endurance Foods|Bengalore|2026-09-19 17:23:...|      customers.csv|               1404|
|     789102|     Endurance Foods|Hyderabad|20

In [0]:
df.bronze.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- _metadata_file_name: string (nullable = true)
 |-- _metadata_file_size: long (nullable = true)



In [0]:
#1.Remove Duplicates
df_duplicates = df.bronze.groupBy("customer_id").count().filter("count > 1")
display(df_duplicates)

customer_id,count
789321,2
789503,2
789522,2
789603,2


In [0]:
print("Rows after duplicates dropped",df.bronze.count())
df_silver = df.bronze.dropDuplicates(['customer_id'])
print("Rows after duplicates dropped",df_silver.count())


Rows after duplicates dropped 39
Rows after duplicates dropped 35


In [0]:
#2.Remove customer name spaces
display(
    df_silver.filter(f.col("customer_name") != f.trim(f.col("customer_name")))
)

customer_id,customer_name,city,read_timestamp,_metadata_file_name,_metadata_file_size
789121,HydroBoost Nutrition,Hyderabad,2026-09-19T17:23:46.620Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-09-19T17:23:46.620Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-09-19T17:23:46.620Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-09-19T17:23:46.620Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-09-19T17:23:46.620Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-09-19T17:23:46.620Z,customers.csv,1404


In [0]:
#Trimming
df_silver = df_silver.withColumn(
    "customer_name", f.trim(f.col("customer_name"))
)

In [0]:
#Checking 
display(
    df_silver.filter(f.col("customer_name") != f.trim(f.col("customer_name")))
)

customer_id,customer_name,city,read_timestamp,_metadata_file_name,_metadata_file_size


In [0]:
#Missing values in city column
df_silver.select('city').distinct().show()

+----------+
|      city|
+----------+
| Bengaluru|
| Hyderabad|
| New Delhi|
| Bengalore|
|Hyderabadd|
|      NULL|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|  NewDheli|
+----------+



In [0]:
#Typos -> correcting names
city_mapping ={
    'Bengaluruu':'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'hyderbad': 'hyderabad',

    'NewDelhee': 'New Delhi',
    'NewDelhi': 'New Delhi',
    'newDhelee': 'New Delhi',

}

allowed = ['Bengaluru', 'Hyderabad', 'New Delhi']

df_silver = (
    df_silver
    .replace(city_mapping,subset=['city'])
    .withColumn(
        'city',
                f.when(f.col("city").isNull(),None)
                .when(f.col("city").isin(allowed),f.col("city"))
                .otherwise(None)
                )

)

#sanity check
df_silver.select('city').distinct().show()

+---------+
|     city|
+---------+
|Bengaluru|
|Hyderabad|
|New Delhi|
|     NULL|
+---------+



In [0]:
#Title Case Fixing
df_silver = df_silver.withColumn(
    "customer_name", 
    f.when(f.col("customer_name").isNull(), None)
    .otherwise(f.initcap(f.col("customer_name")))
)
df_silver.select('customer_name').distinct().show()



+--------------------+
|       customer_name|
+--------------------+
|      Fitfuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|Hydroboost Nutrition|
|Macrobite Superfoods|
|      Powersnack Hub|
|   Sprintx Nutrition|
|    Zenathlete Foods|
|Peak Performance ...|
| Primefuel Nutrition|
|       Recovery Lane|
|      Staminax Store|
|Eliteathlete Nutr...|
|      Gameplan Foods|
|   Champion's Choice|
+--------------------+



In [0]:
df_silver.filter(f.col("city").isNull()).show(truncate=False)

+-----------+-------------------+----+--------------------------+-------------------+-------------------+
|customer_id|customer_name      |city|read_timestamp            |_metadata_file_name|_metadata_file_size|
+-----------+-------------------+----+--------------------------+-------------------+-------------------+
|789403     |Sprintx Nutrition  |NULL|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789420     |Zenathlete Foods   |NULL|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789421     |Zenathlete Foods   |NULL|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789521     |Primefuel Nutrition|NULL|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789522     |Primefuel Nutrition|NULL|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789603     |Recovery Lane      |NULL|2026-09-19 17:23:46.620332|customers.csv      |1404               |
+-----------+-------------------+----+--------

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(f.col("customer_name").isin(null_customer_names)).show(truncate=False)

+-----------+-------------------+---------+--------------------------+-------------------+-------------------+
|customer_id|customer_name      |city     |read_timestamp            |_metadata_file_name|_metadata_file_size|
+-----------+-------------------+---------+--------------------------+-------------------+-------------------+
|789601     |Recovery Lane      |Bengaluru|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789420     |Zenathlete Foods   |Bengaluru|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789421     |Zenathlete Foods   |NULL     |2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789520     |Primefuel Nutrition|Bengaluru|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789403     |Sprintx Nutrition  |New Delhi|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789521     |Primefuel Nutrition|Hyderabad|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|

In [0]:
#City correction by selfguide
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

customer_id,fixed_city
789403,New Delhi
789420,Bengaluru
789521,Hyderabad
789603,Hyderabad


In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        f.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

In [0]:
# Sanity Checks

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(f.col("customer_name").isin(null_customer_names)).show(truncate=False)

+-----------+-------------------+---------+--------------------------+-------------------+-------------------+
|customer_id|customer_name      |city     |read_timestamp            |_metadata_file_name|_metadata_file_size|
+-----------+-------------------+---------+--------------------------+-------------------+-------------------+
|789601     |Recovery Lane      |Bengaluru|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789420     |Zenathlete Foods   |Bengaluru|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789421     |Zenathlete Foods   |NULL     |2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789520     |Primefuel Nutrition|Bengaluru|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789403     |Sprintx Nutrition  |New Delhi|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|789521     |Primefuel Nutrition|Hyderabad|2026-09-19 17:23:46.620332|customers.csv      |1404               |
|

In [0]:
#Convert customer_id to string
df_silver = df_silver.withColumn("customer_id", f.col("customer_id").cast("string"))
print(df_silver.printSchema())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- _metadata_file_name: string (nullable = true)
 |-- _metadata_file_size: long (nullable = true)

None


In [0]:
#Standarizing with parent company Data Model
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        f.concat_ws("-", "customer_name", f.coalesce(f.col("city"), f.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", f.lit("India"))
    .withColumn("platform", f.lit("Sports Bar"))
    .withColumn("channel", f.lit("Acquisition"))
)

In [0]:
display(df_silver.limit(5))

customer_id,customer_name,city,read_timestamp,_metadata_file_name,_metadata_file_size,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,2026-09-19T17:23:46.620Z,customers.csv,1404,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,2026-09-19T17:23:46.620Z,customers.csv,1404,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-09-19T17:23:46.620Z,customers.csv,1404,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,2026-09-19T17:23:46.620Z,customers.csv,1404,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,2026-09-19T17:23:46.620Z,customers.csv,1404,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition


In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

###Gold Layer Processing ###

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

###Merging Data Source with parent Company Data ###

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    f.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
 delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]